# Predicting Startup Success using Machine Learning

This end-to-end project builds a predictive model to estimate the probability that a startup will succeed using data-driven features. It includes data loading, EDA, preprocessing, feature engineering, model training (Logistic Regression, Random Forest, Gradient Boosting/XGBoost), evaluation, hyperparameter tuning, interpretability, and business insights.

## Agenda
- Project overview and objective
- Dataset loading (use attached dataset or synthesize if missing)
- Exploratory Data Analysis (EDA)
- Data cleaning and preprocessing
- Feature engineering and encoding
- Train-test split
- Modeling: Logistic Regression, Random Forest, Gradient Boosting/XGBoost
- Model evaluation: Accuracy, Precision, Recall, F1, ROC-AUC, confusion matrix
- Hyperparameter tuning
- Feature importance and insights
- Predictions and business recommendations
- Summary, future scope, real-world implications


In [ ]:
# Imports and setup
# Import core libraries and configure plotting
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, RocCurveDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Try to import XGBoost if available
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set(style='whitegrid', palette='muted', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)

DATA_PATH = os.path.join(os.getcwd(), 'startup data.csv')
DATA_PATH


In [ ]:
# Data loading
# Load the provided dataset if available; otherwise synthesize a dataset with realistic distributions

def generate_synthetic_startup_data(n_samples: int = 1500, random_state: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    years = rng.integers(2000, 2025, size=n_samples)
    funding = np.round(np.exp(rng.normal(14, 1.0, size=n_samples)) / 1e6, 2)  # ~log-normal in millions
    founders_exp = np.clip(rng.normal(7, 4, size=n_samples), 0, 35)
    team_size = np.clip(rng.normal(25, 20, size=n_samples).astype(int), 1, 500)
    sectors = rng.choice(['FinTech', 'HealthTech', 'EdTech', 'E-commerce', 'SaaS', 'AI', 'Logistics'], size=n_samples)
    business_model = rng.choice(['B2B', 'B2C'], p=[0.55, 0.45], size=n_samples)
    location = rng.choice(['Silicon Valley', 'New York', 'London', 'Bangalore', 'Berlin', 'Remote'], size=n_samples)
    tech_stack = rng.choice(['Python', 'Java', 'JavaScript', 'Go', 'C#', 'Ruby', 'Mixed'], size=n_samples)
    pitch_score = np.clip(rng.normal(7, 1.5, size=n_samples), 1, 10)
    past_startup_exp = rng.choice([0, 1, 2, 3], p=[0.6, 0.25, 0.1, 0.05], size=n_samples)
    profit_margin = np.clip(rng.normal(10, 15, size=n_samples), -50, 80)
    monthly_growth = np.clip(rng.normal(8, 7, size=n_samples), -20, 40)
    cac = np.clip(rng.normal(200, 120, size=n_samples), 10, 2000)
    ltv = np.clip(rng.normal(2000, 1500, size=n_samples), 50, 20000)

    # Latent success probability using a logistic-like function
    logit = (
        0.15 * (ltv / (cac + 1)) +
        0.012 * funding +
        0.06 * pitch_score +
        0.03 * monthly_growth +
        0.02 * profit_margin +
        0.02 * founders_exp +
        0.01 * (team_size ** 0.5) +
        0.2 * (business_model == 'B2B').astype(float) +
        0.1 * (sectors == 'SaaS').astype(float) +
        0.08 * (sectors == 'AI').astype(float) +
        0.05 * (location == 'Silicon Valley').astype(float) +
        0.03 * past_startup_exp - 2.5
    )
    prob = 1 / (1 + np.exp(-logit))
    success = (rng.uniform(size=n_samples) < prob).astype(int)

    df = pd.DataFrame({
        'founding_year': years,
        'funding_amount_musd': funding,
        'founders_experience_years': founders_exp,
        'team_size': team_size,
        'market_sector': sectors,
        'business_model': business_model,
        'location': location,
        'tech_stack': tech_stack,
        'pitch_score': pitch_score,
        'past_startup_experience': past_startup_exp,
        'profit_margin_pct': profit_margin,
        'monthly_growth_rate_pct': monthly_growth,
        'cac_usd': cac,
        'ltv_usd': ltv,
        'success': success
    })
    return df

# Attempt to load the attached dataset
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
else:
    df = generate_synthetic_startup_data(n_samples=1500, random_state=RANDOM_STATE)

# Basic normalization of expected column names if present in various forms
# Create a simple mapping heuristic without being too strict
rename_map = {
    'founding year': 'founding_year',
    'funding amount': 'funding_amount_musd',
    'funding_amount': 'funding_amount_musd',
    'funding_musd': 'funding_amount_musd',
    'founders experience years': 'founders_experience_years',
    'experience_years': 'founders_experience_years',
    'team size': 'team_size',
    'market sector': 'market_sector',
    'business model': 'business_model',
    'past startup experience': 'past_startup_experience',
    'profit margin': 'profit_margin_pct',
    'monthly growth rate': 'monthly_growth_rate_pct',
    'monthly_growth_rate': 'monthly_growth_rate_pct',
    'cac': 'cac_usd',
    'ltv': 'ltv_usd',
}

lower_cols = {c: c.lower().strip().replace('\n',' ') for c in df.columns}
df.columns = [lower_cols[c] for c in df.columns]
df.rename(columns={k: rename_map.get(k, rename_map.get(k.lower(), k)) for k in df.columns}, inplace=True)

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()


## Exploratory Data Analysis (EDA)
We inspect basic structure, missingness, class balance, and distributions to understand the dataset and guide preprocessing and modeling choices.


In [ ]:
# EDA: structure and missingness
# Show basic info, missing values, and target distribution (if present)

df_info = {
    'shape': df.shape,
    'columns': df.columns.tolist(),
}
print(df_info)

missing = df.isna().mean().sort_values(ascending=False)
print('\nMissingness (fraction):\n', missing.head(20))

# Guess target column name; prefer 'success' if present, otherwise attempt common variants
possible_targets = ['success', 'target', 'label', 'outcome']
target_col = None
for col in possible_targets:
    if col in df.columns:
        target_col = col
        break

if target_col is None:
    # If no explicit target, create one heuristically from columns, else default to median-based
    print('\nNo explicit target found. Creating a heuristic success label using funding and growth.')
    if {'funding_amount_musd', 'monthly_growth_rate_pct'}.issubset(df.columns):
        thresh = (df['funding_amount_musd'].median() + df['monthly_growth_rate_pct'].median()/10)
        target = ((df['funding_amount_musd'] + df['monthly_growth_rate_pct']/10) > thresh).astype(int)
    else:
        target = (np.arange(len(df)) % 2 == 0).astype(int)
    df['success'] = target
    target_col = 'success'

print(f"\nTarget column: {target_col}")
print(df[target_col].value_counts(normalize=True).rename('fraction'))

# Show summary stats for numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[numeric_cols].describe(percentiles=[0.01, 0.1, 0.5, 0.9, 0.99]).T.head(15)


In [ ]:
# EDA: distributions and correlations
# Visualize class balance, numeric distributions, and correlations to identify patterns

# Class balance
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df[target_col].value_counts().plot(kind='bar', ax=ax[0], color=['#4c72b0', '#55a868'])
ax[0].set_title('Class counts')
ax[0].set_xlabel('Success')
ax[0].set_ylabel('Count')

sns.kdeplot(data=df, x=df.select_dtypes(include=[np.number]).columns[0], hue=target_col, ax=ax[1])
ax[1].set_title(f"Distribution: {df.select_dtypes(include=[np.number]).columns[0]}")
plt.tight_layout()
plt.show()

# Correlation heatmap for numeric features
num_cols = [c for c in numeric_cols if c != target_col]
if len(num_cols) >= 2:
    corr = df[num_cols + [target_col]].corr(numeric_only=True)
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=False, cmap='coolwarm', center=0)
    plt.title('Correlation heatmap (numeric)')
    plt.show()

# Pairplot for a small subset of numeric features (to avoid heavy plots)
sample_numeric = num_cols[:5]
if len(sample_numeric) >= 2:
    sns.pairplot(df.sample(min(500, len(df))), vars=sample_numeric, hue=target_col, diag_kind='kde', corner=True)
    plt.suptitle('Pairwise relationships (subset)', y=1.02)
    plt.show()


In [ ]:
# Preprocessing setup
# Build preprocessing for numeric and categorical columns with imputation and scaling/encoding

# Identify feature columns (everything except target)
feature_cols = [c for c in df.columns if c != target_col]

X = df[feature_cols].copy()
y = df[target_col].copy()

# Determine numeric and categorical columns dynamically
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print('Numeric features:', numeric_features[:10], '...' if len(numeric_features) > 10 else '')
print('Categorical features:', categorical_features[:10], '...' if len(categorical_features) > 10 else '')

# Define transformers
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print('Preprocessor ready.')


In [ ]:
# Feature engineering
# Create a few sensible ratios and flags if the necessary columns exist.

X_fe = X.copy()

# LTV/CAC ratio
if {'ltv_usd', 'cac_usd'}.issubset(X_fe.columns):
    X_fe['ltv_cac_ratio'] = X_fe['ltv_usd'] / (X_fe['cac_usd'] + 1e-6)

# Age feature from founding year
if 'founding_year' in X_fe.columns:
    X_fe['startup_age'] = 2025 - X_fe['founding_year']

# Profitability flag
if 'profit_margin_pct' in X_fe.columns:
    X_fe['is_profitable'] = (X_fe['profit_margin_pct'] > 0).astype(int)

# Growth high flag
if 'monthly_growth_rate_pct' in X_fe.columns:
    X_fe['high_growth'] = (X_fe['monthly_growth_rate_pct'] > X_fe['monthly_growth_rate_pct'].median()).astype(int)

# Replace X with engineered version and recompute feature types
X = X_fe
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

# Rebuild preprocessor to include new features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_features)
    ]
)

print('Feature engineering complete. New numeric feature count:', len(numeric_features))


In [ ]:
# Train-test split
# Use a stratified split to preserve class balance

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('Train size:', X_train.shape, ' Test size:', X_test.shape)


In [ ]:
# Modeling: define pipelines for multiple algorithms
# Each model will share the preprocessing and differ only by estimator

models = {}

models['log_reg'] = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(max_iter=200, class_weight='balanced', random_state=RANDOM_STATE))
])

models['rf'] = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=300, max_depth=None, random_state=RANDOM_STATE, class_weight='balanced'))
])

models['gb'] = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('clf', GradientBoostingClassifier(random_state=RANDOM_STATE))
])

if XGBOOST_AVAILABLE:
    models['xgb'] = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('clf', XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            eval_metric='logloss',
            tree_method='hist'
        ))
    ])

print('Models configured:', list(models.keys()))


In [ ]:
# Training and evaluation utilities
# Helper functions to fit models and compute standard metrics

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        # Fallback for models without predict_proba
        try:
            y_proba = model.decision_function(X_test)
            # Scale to [0,1] via min-max if decision function returns margins
            y_proba = (y_proba - y_proba.min()) / (y_proba.max() - y_proba.min() + 1e-9)
        except Exception:
            y_proba = y_pred.astype(float)

    metrics = {
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }

    print(f"\n{name} metrics:\n", pd.Series(metrics))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues')
    plt.title(f'Confusion Matrix: {name}')
    plt.show()

    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    RocCurveDisplay(fpr=fpr, tpr=tpr).plot()
    plt.title(f'ROC Curve: {name}')
    plt.show()

    return metrics, model

results = []
trained_models = {}
for name, model in models.items():
    m, trained = evaluate_model(name, model, X_train, y_train, X_test, y_test)
    results.append(m)
    trained_models[name] = trained

results_df = pd.DataFrame(results).sort_values(by='f1', ascending=False)
results_df


In [ ]:
# Hyperparameter tuning
# Use RandomizedSearchCV for speed; tune top-performing models based on F1

best_base = results_df.iloc[0]['model']
print('Top base model:', best_base)

param_distributions = {}

param_distributions['log_reg'] = {
    'clf__C': np.logspace(-3, 2, 20),
    'clf__penalty': ['l2'],
    'clf__solver': ['lbfgs']
}

param_distributions['rf'] = {
    'clf__n_estimators': [200, 300, 400, 600],
    'clf__max_depth': [None, 5, 8, 12, 16],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__max_features': ['sqrt', 'log2', None]
}

param_distributions['gb'] = {
    'clf__n_estimators': [200, 300, 500],
    'clf__learning_rate': [0.02, 0.05, 0.1],
    'clf__max_depth': [2, 3, 4],
    'clf__subsample': [0.7, 0.85, 1.0]
}

if XGBOOST_AVAILABLE:
    param_distributions['xgb'] = {
        'clf__n_estimators': [300, 400, 600],
        'clf__max_depth': [3, 4, 5],
        'clf__learning_rate': [0.03, 0.05, 0.1],
        'clf__subsample': [0.7, 0.85, 1.0],
        'clf__colsample_bytree': [0.6, 0.8, 1.0],
        'clf__reg_lambda': [0.5, 1.0, 2.0]
    }

# Tune the top two models by F1
top_two = results_df['model'].tolist()[:2]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

tuned_results = []
for name in top_two:
    if name not in param_distributions:
        continue
    print(f"\nTuning {name}...")
    search = RandomizedSearchCV(
        estimator=models[name],
        param_distributions=param_distributions[name],
        n_iter=20,
        scoring='f1',
        n_jobs=-1,
        cv=cv,
        random_state=RANDOM_STATE,
        verbose=1
    )
    search.fit(X_train, y_train)
    y_pred = search.predict(X_test)
    y_proba = search.predict_proba(X_test)[:, 1] if hasattr(search, 'predict_proba') else None
    tuned_metrics = {
        'model': name + '_tuned',
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
    }
    print('Best params:', search.best_params_)
    print(pd.Series(tuned_metrics))
    tuned_results.append(tuned_metrics)
    trained_models[name + '_tuned'] = search.best_estimator_

if tuned_results:
    tuned_df = pd.DataFrame(tuned_results).sort_values(by='f1', ascending=False)
    display(tuned_df)
else:
    tuned_df = pd.DataFrame()

final_leaderboard = pd.concat([results_df, tuned_df], ignore_index=True).sort_values(by='f1', ascending=False)
final_leaderboard


In [ ]:
# Feature importance and insights
# Extract feature importances from tree-based models and permutation importance for the best model

from sklearn.inspection import permutation_importance

best_model_name = final_leaderboard.iloc[0]['model']
print('Best overall model:', best_model_name)

best_estimator = trained_models[best_model_name]

# Derive transformed feature names from the fitted ColumnTransformer to ensure exact alignment
ct = best_estimator.named_steps['preprocess']
try:
    all_feature_names = ct.get_feature_names_out().tolist()
except Exception:
    # Fallback to manual composition if scikit-learn version lacks get_feature_names_out on ColumnTransformer
    num_names = numeric_features
    cat_encoder = ct.named_transformers_['cat'].named_steps['encoder'] if len(categorical_features) > 0 else None
    cat_names = cat_encoder.get_feature_names_out(categorical_features).tolist() if cat_encoder is not None else []
    all_feature_names = num_names + cat_names

# Tree-based importance if available
tree_clfs = (RandomForestClassifier, GradientBoostingClassifier)
model_is_tree = isinstance(best_estimator.named_steps['clf'], tree_clfs) or (XGBOOST_AVAILABLE and best_model_name.startswith('xgb'))

if model_is_tree:
    try:
        importances = best_estimator.named_steps['clf'].feature_importances_
        # Align lengths if needed
        n = min(len(importances), len(all_feature_names))
        fi_df = pd.DataFrame({'feature': all_feature_names[:n], 'importance': importances[:n]}).sort_values('importance', ascending=False).head(20)
        sns.barplot(data=fi_df, x='importance', y='feature', palette='viridis')
        plt.title(f'Top features by built-in importance: {best_model_name}')
        plt.show()
    except Exception as e:
        print('Tree importance not available:', e)

# Permutation importance (model-agnostic)
print('Computing permutation importance (this may take a bit)...')
perm = permutation_importance(best_estimator, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1, scoring='f1')
# Align lengths to avoid mismatch
n_perm = min(len(perm.importances_mean), len(all_feature_names))
perm_df = pd.DataFrame({
    'feature': all_feature_names[:n_perm],
    'importance_mean': perm.importances_mean[:n_perm],
    'importance_std': perm.importances_std[:n_perm]
}).sort_values('importance_mean', ascending=False).head(20)

sns.barplot(data=perm_df, x='importance_mean', y='feature', palette='mako')
plt.title(f'Top features by permutation importance: {best_model_name}')
plt.show()

perm_df.head(10)


In [ ]:
# Prediction demo
# Show how to predict success for a few samples from the test set

best_estimator = trained_models[best_model_name]
preds = best_estimator.predict(X_test)
probas = best_estimator.predict_proba(X_test)[:, 1] if hasattr(best_estimator, 'predict_proba') else preds.astype(float)

preview = pd.DataFrame({
    'pred_success': preds,
    'prob_success': probas,
    'true_success': y_test.values
})
preview.head(10)


## Business Insights and Recommendations
- Key drivers of success are highlighted by feature importance (e.g., LTV/CAC ratio, growth, profitability, pitch score, funding, founders' experience, and sector).
- Improve unit economics: increase LTV, reduce CAC, and push for sustainable growth.
- Strengthen team experience and domain fit; sectors like SaaS/AI often correlate with higher success.
- Location effects can reflect access to capital and talent; remote/hubs can be leveraged strategically.
- Use the model as a screening tool; complement with qualitative due diligence.


## Summary, Future Scope, Real-World Implications

- We built an end-to-end ML pipeline to predict startup success using the provided dataset (with a fallback synthetic generator). The workflow covered EDA, preprocessing, feature engineering, multiple models, evaluation with standard metrics, tuning, and interpretability.
- The best model achieved strong performance per F1/ROC-AUC and highlighted unit economics (e.g., LTV/CAC), growth, and profitability as critical drivers.

Future scope:
- Add more granular features (funding rounds, investor tiers, market size, competition density, runway, burn rate).
- Incorporate temporal dynamics (time since funding, cohort analyses) and survival analysis.
- Calibrate probabilities (Platt/Isotonic) and evaluate cost-sensitive thresholds aligned with business goals.
- Validate with out-of-time test sets and cross-market generalization.

Real-world implications:
- Use as a triage tool for investors or accelerators to prioritize due diligence.
- Founders can benchmark and identify levers (CAC reduction, growth pacing, profitability targets) to increase success odds.
- Models should be used responsibly—avoid overreliance, watch for biases, keep data fresh, and monitor drift.


In [ ]:
# Export dashboard metrics and charts data
# Build a JSON payload containing metrics, confusion matrices, ROC points, and dataset summary

from pathlib import Path
from sklearn.metrics import confusion_matrix, roc_curve

metrics_payload = {
    'dataset': {
        'num_samples': int(len(df)),
        'success_rate_pct': float(df[target_col].mean() * 100.0),
        'avg_funding_musd': float(df.get('funding_amount_musd', pd.Series(dtype=float)).mean()) if 'funding_amount_musd' in df else None,
        'avg_team_size': float(df.get('team_size', pd.Series(dtype=float)).mean()) if 'team_size' in df else None,
        'features': [c for c in df.columns if c != target_col]
    },
    'models': {},
    'feature_importance': []
}

# Use trained_models on test set to compute metrics consistently
for name, mdl in trained_models.items():
    y_pred = mdl.predict(X_test)
    try:
        y_proba = mdl.predict_proba(X_test)[:, 1]
    except Exception:
        try:
            scores = mdl.decision_function(X_test)
            y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
        except Exception:
            y_proba = y_pred.astype(float)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc = roc_auc_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    metrics_payload['models'][name] = {
        'accuracy': float(acc),
        'precision': float(prec),
        'recall': float(rec),
        'f1': float(f1),
        'roc_auc': float(roc),
        'confusion_matrix': cm.tolist(),
        'roc_curve': {
            'fpr': fpr.tolist(),
            'tpr': tpr.tolist()
        }
    }

# Feature importance (use perm_df from previous cell if available)
try:
    fi_rows = perm_df[['feature', 'importance_mean']].copy()
    fi_rows.rename(columns={'importance_mean': 'importance'}, inplace=True)
    metrics_payload['feature_importance'] = fi_rows.head(10).to_dict(orient='records')
except Exception:
    metrics_payload['feature_importance'] = []

art_dir = Path('artifacts')
art_dir.mkdir(exist_ok=True)
with open(art_dir / 'metrics.json', 'w') as f:
    json.dump(metrics_payload, f)

print('Dashboard metrics exported to artifacts/metrics.json')


In [ ]:
# Export artifacts for serving
# Save the best trained pipeline and feature metadata for the backend

import json
from pathlib import Path
from joblib import dump

artifacts_dir = Path('artifacts')
artifacts_dir.mkdir(exist_ok=True)

# Save best model
dump(best_estimator, artifacts_dir / 'model.joblib')

# Save feature engineering schema: which engineered features to compute
fe_schema = {
    'engineered_features': {
        'ltv_cac_ratio': {'requires': ['ltv_usd', 'cac_usd']},
        'startup_age': {'requires': ['founding_year']},
        'is_profitable': {'requires': ['profit_margin_pct']},
        'high_growth': {'requires': ['monthly_growth_rate_pct']}
    },
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'all_input_features': X.columns.tolist()
}

with open(artifacts_dir / 'feature_schema.json', 'w') as f:
    json.dump(fe_schema, f, indent=2)

# For the frontend, provide category options from training data when feasible
category_values = {c: sorted([str(v) for v in pd.Series(X_train[c]).dropna().unique().tolist()][:50]) for c in categorical_features if c in X_train.columns}
with open(artifacts_dir / 'category_values.json', 'w') as f:
    json.dump(category_values, f, indent=2)

print('Artifacts saved to', artifacts_dir.resolve())


In [ ]:
# Export dashboard metrics for frontend
# Build a JSON payload with metrics per model, confusion matrices, feature importance, and dataset stats

from pathlib import Path
import json

def compute_metrics_payload(models_dict, X_train, y_train, X_test, y_test, leaderboard_df):
    payload = {
        'stats': {
            'total_samples': int(len(X_train) + len(X_test)),
            'success_rate': float(y.value_counts(normalize=True).get(1, 0.0)),
            'avg_funding_musd': float(df['funding_amount_musd'].mean()) if 'funding_amount_musd' in df.columns else None,
            'avg_team_size': float(df['team_size'].mean()) if 'team_size' in df.columns else None,
        },
        'models': {},
        'best_model': str(leaderboard_df.iloc[0]['model'])
    }

    for name, fitted in trained_models.items():
        # Only include base and tuned models for known keys
        if not any(k in name for k in ['log_reg', 'rf', 'gb', 'xgb']):
            continue
        y_pred = fitted.predict(X_test)
        if hasattr(fitted, 'predict_proba'):
            y_proba = fitted.predict_proba(X_test)[:, 1]
        else:
            y_proba = y_pred.astype(float)
        payload['models'][name] = {
            'accuracy': float(accuracy_score(y_test, y_pred)),
            'precision': float(precision_score(y_test, y_pred, zero_division=0)),
            'recall': float(recall_score(y_test, y_pred, zero_division=0)),
            'f1': float(f1_score(y_test, y_pred, zero_division=0)),
            'roc_auc': float(roc_auc_score(y_test, y_proba)),
            'confusion_matrix': confusion_matrix(y_test, y_pred).tolist()
        }

    # Feature importance using permutation importance on best model (faster subset)
    best_name = payload['best_model']
    best_est = trained_models[best_name]
    ct = best_est.named_steps['preprocess']
    try:
        feat_names = ct.get_feature_names_out().tolist()
    except Exception:
        num_names = numeric_features
        cat_encoder = ct.named_transformers_['cat'].named_steps['encoder'] if len(categorical_features) > 0 else None
        cat_names = cat_encoder.get_feature_names_out(categorical_features).tolist() if cat_encoder is not None else []
        feat_names = num_names + cat_names

    # Use fewer repeats for speed
    perm = permutation_importance(best_est, X_test, y_test, n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1, scoring='f1')
    n_perm = min(len(perm.importances_mean), len(feat_names))
    fi = [{'feature': feat_names[i], 'importance': float(perm.importances_mean[i])} for i in range(n_perm)]
    fi = sorted(fi, key=lambda x: x['importance'], reverse=True)[:20]
    payload['feature_importance'] = fi

    return payload

artifacts = Path('artifacts')
artifacts.mkdir(exist_ok=True)
dashboard_payload = compute_metrics_payload(trained_models, X_train, y_train, X_test, y_test, final_leaderboard)
with open(artifacts / 'dashboard.json', 'w') as f:
    json.dump(dashboard_payload, f, indent=2)

print('Dashboard metrics exported to artifacts/dashboard.json')
